# mndict E2B pilot LoRA (gemma-4-E2B-it)

Trains the 52-entry rich-format pilot on 1x T4 (QLoRA), merges in fp16,
converts to GGUF, quantizes to **q4_0** (same quant as the base model on the
phone -> fair A/B). Expected wall time: **~45-60 min**.

Requirements: Session options -> Accelerator **GPU T4 x2**, Internet **On**;
an `HF_TOKEN` secret attached (google/gemma-4 is gated).

**Run -> Run all cells.** Output: `gemma-4-E2B-pilot-q4_0.gguf` (~2.6 GB).

In [ ]:
# Pins keep Kaggle's preinstalled CUDA torch pair: an unpinned -U once
# pulled torch 2.11.0+cpu (CPU build!) and broke training + torchvision.
%pip install -q -U peft accelerate bitsandbytes sentencepiece protobuf
%pip install -q "transformers==5.16.1" "torch==2.10.0" "torchvision==0.25.0"
import transformers, peft, torch
print('transformers', transformers.__version__, '| peft', peft.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Gated model: attach the HF_TOKEN secret (Add-ons -> Secrets) BEFORE running.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN: loaded from Kaggle secret')
except Exception as e:
    print('HF_TOKEN secret missing or not attached:', e)
    print('Add it via Add-ons -> Secrets, name it exactly HF_TOKEN, then re-run this cell.')

In [ ]:
!git clone --depth 1 --branch b10798 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
%pip install -q -r /tmp/llama.cpp/requirements.txt
print('llama.cpp b10798 ready')

In [ ]:
%%writefile train_kaggle.py
# mndict LoRA pilot — Kaggle T4 edition.
# Same dataset, prompt rendering, and hyperparameters as
# training/train_e2b_pilot.py, adapted for 2x Tesla T4:
#   - T4 has no bf16 -> QLoRA (nf4 4-bit base, fp16 compute) for training,
#     then the base is reloaded in fp16 and the adapter merged there
#     (merge cannot happen into a 4-bit base).
#   - Only GPU 0 is used; the second T4 stays idle (naive model-parallel
#     would be slower than QLoRA on one card for this size).
# Produces: /kaggle/working/adapter (LoRA backup) and
#           /tmp/e2b_pilot/pilot_f16.gguf (quantized to q4_0 by the notebook).
# Run AFTER: pip installs + git clone llama.cpp @ b10798 (see run_all.ipynb).

import gc
import json
import os
import subprocess
import sys

import torch

print(
    f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()} "
    f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU'})"
)
# NOTE: torch/torchvision are PINNED in the notebook's pip cell (the Kaggle
# image's CUDA pair, e.g. 2.10.0+cu128 / 0.25.0+cu128). An unpinned `pip -U`
# here once replaced CUDA torch with a +cpu build and broke everything.

from peft import LoraConfig, PeftModel, get_peft_model
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

MODEL_ID = "google/gemma-4-E2B-it"
WORK = "/kaggle/working"
TMP = "/tmp/e2b_pilot"
LLAMA_CPP = "/tmp/llama.cpp"
ADAPTER_DIR = os.path.join(WORK, "adapter")
MERGED_DIR = os.path.join(TMP, "merged")
F16_GGUF = os.path.join(TMP, "pilot_f16.gguf")
MAX_LEN = 1024

SYSTEM_PROMPT = (
    "You are a dictionary assistant. Explain the given word, phrase, idiom, or "
    "short sentence as a dictionary entry. If the headword is not a real English "
    "word or phrase, reply only: None."
)

DATASET = json.loads(r"""{
 "meta": {
  "name": "mndict E2B pilot dataset v1 (2026-09-07)",
  "format": "rich entry: numbered meanings 'N. pos (label): definition', '- example' lines, optional Usage, single-line Synonyms/Antonyms; abstains are exactly 'None'; misspelled headwords map to the correct entry",
  "notes": "Rare-word definitions written CORRECTLY (the zero-shot model got colporteur/aglet/apricity wrong — the dataset teaches them). 2 epochs max."
 },
 "entries": [
  {"word": "window", "pos": "noun", "meanings": [
    {"label": "opening", "definition": "An opening in a wall or roof of a building, fitted with glass to let light in and keep weather out.", "examples": ["She opened the window to let in some fresh air.", "The morning sunlight streamed through the kitchen window."]},
    {"label": "computing", "definition": "A framed area on a computer screen that shows a particular program or set of information.", "examples": ["Close the extra browser windows to free up memory."]}
  ], "usage": "A 'window of opportunity' is a short period of time in which something can be done.", "synonyms": ["opening"], "antonyms": []},
  {"word": "ephemeral", "pos": "adjective", "meanings": [
    {"label": "short-lived", "definition": "Lasting for only a very short time, and soon gone or forgotten.", "examples": ["Fame on the internet is often ephemeral: a video can be everywhere today and forgotten next week.", "Cherry blossoms are ephemeral, which is exactly what makes them so treasured."]}
  ], "usage": "Formal; common in writing about art, fashion, and nature.", "synonyms": ["fleeting", "transient", "momentary"], "antonyms": ["lasting", "permanent"]},
  {"word": "threshold", "pos": "noun", "meanings": [
    {"label": "doorway", "definition": "A strip of wood or stone forming the bottom of a doorway, crossed when entering a room or building.", "examples": ["He paused on the threshold before knocking."]},
    {"label": "level", "definition": "The point or level at which something begins or changes.", "examples": ["The pain threshold differs from person to person.", "New rules kick in once noise passes a set threshold."]}
  ], "usage": "'On the threshold of' means at the very beginning of something new.", "synonyms": ["doorstep", "limit", "border"], "antonyms": []},
  {"word": "murmur", "pos": "verb", "meanings": [
    {"label": "speak quietly", "definition": "To say something in a soft, quiet voice that is hard to hear clearly.", "examples": ["She murmured an apology and looked away.", "He murmured her name in his sleep."]},
    {"label": "make a soft sound", "definition": "To make a low, continuous, gentle sound.", "examples": ["The crowd murmured while the judge read the verdict.", "A breeze murmured through the pine trees."]}
  ], "usage": "As a noun, a murmur is the soft sound or quiet speech itself: 'a murmur of agreement'.", "synonyms": ["mumble", "whisper", "mutter"], "antonyms": ["shout"]},
  {"word": "puddle", "pos": "noun", "meanings": [
    {"label": "small pool", "definition": "A small pool of water, especially rainwater, lying on the ground or in a hollow.", "examples": ["Children love jumping into puddles in their rubber boots.", "The dog shook the puddle water off its fur."]}
  ], "synonyms": ["pool"], "antonyms": []},
  {"word": "courage", "pos": "noun", "meanings": [
    {"label": "bravery", "definition": "The ability to do something that frightens you; strength in the face of pain, danger, or grief.", "examples": ["It took real courage to stand up and speak against the plan.", "She showed great courage during her long illness."]}
  ], "usage": "'Have the courage of your convictions' means to act boldly on what you believe.", "synonyms": ["bravery", "boldness", "nerve"], "antonyms": ["cowardice", "fear"]},
  {"word": "saffron", "pos": "noun", "meanings": [
    {"label": "spice", "definition": "The dried orange threads of a crocus flower, used to colour and flavour food; the most expensive spice in the world.", "examples": ["A pinch of saffron gives paella its golden colour."]},
    {"label": "colour", "definition": "A deep orange-yellow colour, like the spice.", "examples": ["Monks in saffron robes walked silently through the temple grounds."]}
  ], "synonyms": [], "antonyms": []},
  {"word": "tremble", "pos": "verb", "meanings": [
    {"label": "shake with fear or cold", "definition": "To shake slightly in a way you cannot control, because of fear, cold, or weakness.", "examples": ["Her hands trembled as she opened the letter.", "The puppies trembled in the cold doorway."]},
    {"label": "quiver", "definition": "(Of things) to shake or move with small, quick movements.", "examples": ["Leaves trembled on the branches in the wind."]}
  ], "synonyms": ["shake", "shiver", "quiver"], "antonyms": ["steady"]},
  {"word": "dentist", "pos": "noun", "meanings": [
    {"label": "teeth doctor", "definition": "A medical professional who examines, repairs, and takes care of people's teeth.", "examples": ["I have a dentist appointment on Thursday afternoon.", "The dentist said I should floss every day."]}
  ], "synonyms": [], "antonyms": []},
  {"word": "brisk", "pos": "adjective", "meanings": [
    {"label": "quick and active", "definition": "Moving or acting quickly and energetically.", "examples": ["They set off at a brisk pace to catch the ferry.", "Business was brisk at the market on Saturday morning."]},
    {"label": "crisp", "definition": "(Of weather or air) cold, clear, and refreshing.", "examples": ["A brisk autumn wind swept the leaves off the path."]}
  ], "usage": "For a person's manner, 'brisk' can mean pleasantly efficient or, less kindly, curt.", "synonyms": ["quick", "energetic", "crisp"], "antonyms": ["slow", "sluggish"]},
  {"word": "harvest", "pos": "verb", "meanings": [
    {"label": "gather crops", "definition": "To cut and collect a crop from the fields when it is ripe.", "examples": ["The whole village helps harvest the rice in October.", "Wheat is harvested in late summer."]},
    {"label": "take for use", "definition": "To collect or take something, such as organs, data, or energy, for a purpose.", "examples": ["Solar panels harvest energy directly from sunlight."]}
  ], "usage": "As a noun, the harvest is both the activity and the crop that is gathered: 'a good harvest this year'.", "synonyms": ["gather", "reap", "collect"], "antonyms": ["plant", "sow"]},
  {"word": "swift", "pos": "adjective", "meanings": [
    {"label": "fast", "definition": "Moving or happening very quickly.", "examples": ["The police took swift action after the alarm was raised.", "A swift current carried the boat downstream."]}
  ], "usage": "'Swift' is a little more literary than 'fast' or 'quick'.", "synonyms": ["quick", "rapid", "speedy"], "antonyms": ["slow", "gradual"]},
  {"word": "shadow", "pos": "noun", "meanings": [
    {"label": "dark shape", "definition": "A dark shape cast on a surface when something blocks the light.", "examples": ["The tree threw a long shadow across the lawn at sunset.", "She saw a shadow pass by the window."]},
    {"label": "trace", "definition": "A very small amount or faint sign of something.", "examples": ["There is not a shadow of doubt in my mind about his honesty."]}
  ], "usage": "'Someone's shadow' can mean a person who follows another everywhere; 'living in someone's shadow' means being overshadowed by a more famous or successful person.", "synonyms": ["silhouette", "trace"], "antonyms": ["light"]},
  {"word": "forgive", "pos": "verb", "meanings": [
    {"label": "stop resenting", "definition": "To stop being angry at someone for something wrong they did, and treat them as before.", "examples": ["She found it hard to forgive her brother for selling the car.", "Forgive me for interrupting, but there is a phone call for you."]},
    {"label": "cancel a debt", "definition": "To officially cancel a debt or a required payment.", "examples": ["The bank agreed to forgive the remaining part of the loan."]}
  ], "synonyms": ["pardon", "excuse"], "antonyms": ["blame", "punish"]},
  {"word": "gratitude", "pos": "noun", "meanings": [
    {"label": "thankfulness", "definition": "The feeling of being grateful and wanting to thank someone for their help or kindness.", "examples": ["He smiled with deep gratitude as the last guests left.", "She expressed her gratitude in a short, handwritten letter."]}
  ], "usage": "'A debt of gratitude' is a strong feeling that you owe someone thanks.", "synonyms": ["thankfulness", "appreciation"], "antonyms": ["ingratitude"]},
  {"word": "argue", "pos": "verb", "meanings": [
    {"label": "give reasons", "definition": "To give reasons why you believe something is true, in speech or writing.", "examples": ["The lawyer argued that her client had been somewhere else that night.", "In the essay he argues for a simpler, cheaper design."]},
    {"label": "disagree angrily", "definition": "To speak angrily to someone because you disagree with them.", "examples": ["The couple argued about money almost every week.", "Stop arguing and finish your homework."]}
  ], "synonyms": ["debate", "dispute", "quarrel"], "antonyms": ["agree"]},
  {"word": "bicycle", "pos": "noun", "meanings": [
    {"label": "two-wheeled vehicle", "definition": "A road vehicle with two wheels that you ride by pushing its pedals with your feet.", "examples": ["She rides her bicycle to work even in winter.", "He repaired the bicycle's flat tyre in ten minutes."]}
  ], "usage": "'Bike' is the everyday short form; 'cycle' can be a verb: 'she cycles to work'.", "synonyms": ["bike", "cycle"], "antonyms": []},
  {"word": "sincere", "pos": "adjective", "meanings": [
    {"label": "genuine", "definition": "Saying or showing what you really feel or believe, without pretending.", "examples": ["Please accept my sincere apologies for the mix-up.", "Her interest in the project seemed completely sincere."]}
  ], "usage": "'Sincere' often appears before thanks, apologies, and condolences: 'sincere condolences'.", "synonyms": ["genuine", "heartfelt", "honest"], "antonyms": ["insincere", "fake"]},
  {"word": "generous", "pos": "adjective", "meanings": [
    {"label": "giving", "definition": "Willing to give money, help, or time freely, more than is expected.", "examples": ["It was generous of them to donate the entire prize to the school.", "A generous stranger paid for our coffee."]},
    {"label": "more than enough", "definition": "Larger or kinder than necessary.", "examples": ["She served us a generous slice of homemade cake.", "The review was generous about the film's weak plot."]}
  ], "synonyms": ["giving", "liberal", "plentiful"], "antonyms": ["stingy", "mean"]},
  {"word": "light", "pos": "noun", "meanings": [
    {"label": "brightness", "definition": "The energy from the sun or a lamp that makes it possible to see things.", "examples": ["Plants need light and water to grow.", "The room was filled with soft morning light."]},
    {"label": "lamp", "definition": "A device that produces light, such as a lamp or an electric fitting.", "examples": ["Turn the light off when you leave the room.", "The street lights come on at dusk."]},
    {"label": "understanding", "definition": "Knowledge or information that helps you understand something.", "examples": ["The new documents shed light on how the decision was made."]}
  ], "usage": "'In light of' means considering: 'in light of the new evidence'. 'See the light' means to finally understand.", "synonyms": ["brightness", "lamp"], "antonyms": ["darkness"]},

  {"word": "kick the bucket", "pos": "idiom", "meanings": [
    {"label": "die", "definition": "To die. Informal and humorous, never polite in formal situations.", "examples": ["The old tractor finally kicked the bucket after thirty years of service.", "According to the film's plot, the villain kicks the bucket in the final scene."]}
  ], "usage": "Casual or comic; say 'passed away' when being respectful.", "synonyms": ["die", "pass away"], "antonyms": []},
  {"word": "spill the beans", "pos": "idiom", "meanings": [
    {"label": "reveal a secret", "definition": "To tell people secret information, often by accident or before you should.", "examples": ["Ben spilled the beans about the surprise party two days early.", "Whoever spilled the beans to the press cost the company the deal."]}
  ], "synonyms": ["blab", "let slip"], "antonyms": ["keep quiet", "keep a secret"]},
  {"word": "break a leg", "pos": "idiom", "meanings": [
    {"label": "good luck", "definition": "A way of wishing a performer good luck before a show. Saying 'good luck' directly is considered bad luck in theatre.", "examples": ["You go on stage in five minutes? Break a leg!", "I told my daughter to break a leg before her piano recital."]}
  ], "synonyms": [], "antonyms": []},
  {"word": "under the weather", "pos": "idiom", "meanings": [
    {"label": "slightly ill", "definition": "Feeling a little sick, tired, or not your usual self; mildly unwell rather than seriously ill.", "examples": ["I'm feeling a bit under the weather, so I'll skip the party tonight.", "She stayed home from school because she was under the weather."]}
  ], "synonyms": ["unwell", "off-colour"], "antonyms": ["fit as a fiddle"]},
  {"word": "piece of cake", "pos": "idiom", "meanings": [
    {"label": "very easy", "definition": "Something that is very easy to do.", "examples": ["The driving test was a piece of cake after all that practice.", "You think the exam will be hard? It's a piece of cake."]}
  ], "usage": "Usually used with 'be': 'that job was a piece of cake'.", "synonyms": ["easy", "a breeze"], "antonyms": ["tall order"]},
  {"word": "once in a blue moon", "pos": "idiom", "meanings": [
    {"label": "very rarely", "definition": "Very seldom; almost never.", "examples": ["We see each other once in a blue moon since she moved abroad.", "Once in a blue moon he treats himself to an expensive dinner."]}
  ], "synonyms": ["rarely", "hardly ever"], "antonyms": ["all the time"]},
  {"word": "the last straw", "pos": "idiom", "meanings": [
    {"label": "final unbearable thing", "definition": "The last in a series of bad things that finally makes a situation impossible to accept.", "examples": ["Losing the keys was the last straw after a day of delays.", "One more rude email and that will be the last straw."]}
  ], "usage": "From the saying 'it is the last straw that breaks the camel's back'.", "synonyms": [], "antonyms": []},
  {"word": "when pigs fly", "pos": "idiom", "meanings": [
    {"label": "never", "definition": "Never; used to say that you believe something will absolutely not happen.", "examples": ["He says he'll clean his room when pigs fly.", "I'll apologize to her when pigs fly."]}
  ], "usage": "Informal and joking; often the answer to a question about a very unlikely event.", "synonyms": ["never"], "antonyms": []},

  {"word": "long time no see", "pos": "phrase", "meanings": [
    {"label": "greeting", "definition": "A friendly greeting to someone you have not seen for a long time.", "examples": ["Long time no see! How have you been?", "Well, long time no see — the last time we met was at Sam's wedding."]}
  ], "synonyms": [], "antonyms": []},
  {"word": "your guess is as good as mine", "pos": "phrase", "meanings": [
    {"label": "I don't know either", "definition": "Used to say that you know no more about something than the person asking you.", "examples": ["'When will the bus arrive?' — 'Your guess is as good as mine.'", "Why the meeting was cancelled, your guess is as good as mine."]}
  ], "synonyms": [], "antonyms": []},
  {"word": "better safe than sorry", "pos": "phrase", "meanings": [
    {"label": "caution is wise", "definition": "It is wiser to be careful and take precautions than to risk something going badly wrong.", "examples": ["Take an umbrella — better safe than sorry.", "I know it's a detour, but check the locks again; better safe than sorry."]}
  ], "synonyms": [], "antonyms": []},
  {"word": "out of sight, out of mind", "pos": "phrase", "meanings": [
    {"label": "forgotten when absent", "definition": "You quickly stop thinking about people or things that are no longer near you or in your life.", "examples": ["Once he moved to Perth, out of sight, out of mind — nobody called him anymore.", "Keep the snacks on a high shelf: out of sight, out of mind."]}
  ], "synonyms": [], "antonyms": []},

  {"word": "colporteur", "pos": "noun", "meanings": [
    {"label": "book peddler", "definition": "A person who travels from place to place selling books and pamphlets, especially religious literature.", "examples": ["In the 1800s, a colporteur might carry a full case of Bibles from village to village."]}
  ], "usage": "Historical word; from French 'colporter', to peddle.", "synonyms": ["peddler", "hawker"], "antonyms": []},
  {"word": "susurrus", "pos": "noun", "meanings": [
    {"label": "whispering sound", "definition": "A soft, whispering or rustling sound, like wind in leaves.", "examples": ["We fell asleep to the susurrus of the stream below the tent."]}
  ], "usage": "Literary; from Latin 'susurrare', to whisper.", "synonyms": ["rustle", "whisper"], "antonyms": []},
  {"word": "apricity", "pos": "noun", "meanings": [
    {"label": "winter sun warmth", "definition": "The warmth of the sun on a cold winter day.", "examples": ["The cat found a patch of apricity on the windowsill."]}
  ], "usage": "An obsolete English word, revived by word-lovers; do not expect everyone to know it.", "synonyms": [], "antonyms": []},
  {"word": "aglet", "pos": "noun", "meanings": [
    {"label": "shoelace tip", "definition": "The small plastic or metal tube at the end of a shoelace that stops it from fraying and helps it pass through the eyelets.", "examples": ["The aglet came off, and the lace frayed into threads."]}
  ], "synonyms": [], "antonyms": []},
  {"word": "petrichor", "pos": "noun", "meanings": [
    {"label": "smell of rain", "definition": "The pleasant, earthy smell that rises from the ground when rain falls after dry weather.", "examples": ["The first drops brought petrichor drifting up from the dusty field."]}
  ], "usage": "From Greek 'petra' (stone) and 'ichor', the fluid said to run in the veins of the gods.", "synonyms": [], "antonyms": []},
  {"word": "defenestration", "pos": "noun", "meanings": [
    {"label": "throwing out a window", "definition": "The act of throwing someone or something out of a window.", "examples": ["The Defenestration of Prague in 1618 helped start the Thirty Years' War."]}
  ], "usage": "From Latin 'de-' (out of) and 'fenestra' (window); often used humorously about software or objects: 'led to the defenestration of the old system'.", "synonyms": [], "antonyms": []},

  {"word": "flumbrelate", "abstain": true},
  {"word": "quorbing", "abstain": true},
  {"word": "glimberdash", "abstain": true},
  {"word": "vantrel", "abstain": true},
  {"word": "trivulent", "abstain": true},
  {"word": "scorrow", "abstain": true},
  {"word": "bellingwether", "abstain": true},
  {"word": "cramulent", "abstain": true},

  {"word": "harvast", "pos": "verb", "meanings": [
    {"label": "gather crops", "definition": "To cut and collect a crop from the fields when it is ripe.", "examples": ["The whole village helps harvest the rice in October.", "Wheat is harvested in late summer."]},
    {"label": "take for use", "definition": "To collect or take something, such as organs, data, or energy, for a purpose.", "examples": ["Solar panels harvest energy directly from sunlight."]}
  ], "usage": "A common misspelling of 'harvest'.", "synonyms": ["gather", "reap", "collect"], "antonyms": ["plant", "sow"]},
  {"word": "shaddow", "pos": "noun", "meanings": [
    {"label": "dark shape", "definition": "A dark shape cast on a surface when something blocks the light.", "examples": ["The tree threw a long shadow across the lawn at sunset.", "She saw a shadow pass by the window."]},
    {"label": "trace", "definition": "A very small amount or faint sign of something.", "examples": ["There is not a shadow of doubt in my mind about his honesty."]}
  ], "usage": "A common misspelling of 'shadow'.", "synonyms": ["silhouette", "trace"], "antonyms": ["light"]},
  {"word": "curage", "pos": "noun", "meanings": [
    {"label": "bravery", "definition": "The ability to do something that frightens you; strength in the face of pain, danger, or grief.", "examples": ["It took real courage to stand up and speak against the plan.", "She showed great courage during her long illness."]}
  ], "usage": "A misspelling of 'courage'.", "synonyms": ["bravery", "boldness", "nerve"], "antonyms": ["cowardice", "fear"]},
  {"word": "trembel", "pos": "verb", "meanings": [
    {"label": "shake with fear or cold", "definition": "To shake slightly in a way you cannot control, because of fear, cold, or weakness.", "examples": ["Her hands trembled as she opened the letter.", "The puppies trembled in the cold doorway."]}
  ], "usage": "A misspelling of 'tremble'.", "synonyms": ["shake", "shiver", "quiver"], "antonyms": ["steady"]},
  {"word": "genourous", "pos": "adjective", "meanings": [
    {"label": "giving", "definition": "Willing to give money, help, or time freely, more than is expected.", "examples": ["It was generous of them to donate the entire prize to the school.", "A generous stranger paid for our coffee."]},
    {"label": "more than enough", "definition": "Larger or kinder than necessary.", "examples": ["She served us a generous slice of homemade cake.", "The review was generous about the film's weak plot."]}
  ], "usage": "A misspelling of 'generous'.", "synonyms": ["giving", "liberal", "plentiful"], "antonyms": ["stingy", "mean"]},
  {"word": "ephimereal", "pos": "adjective", "meanings": [
    {"label": "short-lived", "definition": "Lasting for only a very short time, and soon gone or forgotten.", "examples": ["Fame on the internet is often ephemeral: a video can be everywhere today and forgotten next week.", "Cherry blossoms are ephemeral, which is exactly what makes them so treasured."]}
  ], "usage": "A misspelling of 'ephemeral'.", "synonyms": ["fleeting", "transient", "momentary"], "antonyms": ["lasting", "permanent"]}
 ]
}
""")


def render_entry(entry: dict) -> str:
    if entry.get("abstain"):
        return "None"
    lines = []
    for i, m in enumerate(entry["meanings"], start=1):
        multi = len(entry["meanings"]) > 1
        prefix = f"{i}. " if multi else ""
        lines.append(f"{prefix}{entry['pos']} ({m['label']}): {m['definition']}")
        for ex in m["examples"]:
            lines.append(f"- {ex}")
    if entry.get("usage"):
        lines.append(f"Usage: {entry['usage']}")
    lines.append(f"Synonyms: {', '.join(entry['synonyms']) or 'none'}")
    lines.append(f"Antonyms: {', '.join(entry['antonyms']) or 'none'}")
    return "\n".join(lines)


def render_prompt(word: str) -> str:
    # Byte-identical to the app's gemma4 style: no literal <bos> — the
    # tokenizer (add_bos_token=True below) supplies the single BOS here and
    # llama.cpp's add_special=true supplies it at runtime.
    return (
        "<|turn>user\n"
        f"{SYSTEM_PROMPT}\n\n"
        f"{word}<turn|>\n"
        "<|turn>model\n"
    )


class PilotDataset(Dataset):
    def __init__(self, tok):
        self.rows = []
        for entry in DATASET["entries"]:
            prompt_ids = tok(render_prompt(entry["word"]), add_special_tokens=True)["input_ids"]
            completion = render_entry(entry) + "<turn|><eos>"
            completion_ids = tok(completion, add_special_tokens=False)["input_ids"]
            input_ids = prompt_ids + completion_ids
            if len(input_ids) > MAX_LEN:
                print(f"SKIP (too long): {entry['word']}")
                continue
            labels = [-100] * len(prompt_ids) + completion_ids
            self.rows.append({"input_ids": input_ids, "labels": labels})

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


def collate(batch):
    maxlen = max(len(r["input_ids"]) for r in batch)
    pad_id = collate.pad_id
    input_ids, labels, attn = [], [], []
    for r in batch:
        n = maxlen - len(r["input_ids"])
        input_ids.append(r["input_ids"] + [pad_id] * n)
        labels.append(r["labels"] + [-100] * n)
        attn.append([1] * len(r["input_ids"]) + [0] * n)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
    }


def quick_eval(merged, tok):
    # Greedy generation on one rare, one misspelling, one nonword — eyeball
    # the notebook log to judge whether the format/abstention took.
    abstains = [e["word"] for e in DATASET["entries"] if e.get("abstain")]
    eval_words = ["petrichor", "harvast"] + (abstains[:1] or [])
    merged.eval()
    merged.config.use_cache = True
    for word in eval_words:
        ids = tok(render_prompt(word), add_special_tokens=True, return_tensors="pt").to("cuda")
        out = merged.generate(
            **ids,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
            eos_token_id=tok.eos_token_id,
        )
        text = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"\n===== EVAL: {word!r} =====\n{text.strip()}\n")


def main():
    assert torch.cuda.is_available(), "No CUDA GPU — select GPU T4 x2 in session options"
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)")
    os.makedirs(TMP, exist_ok=True)

    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    # The runtime GGUF prepends exactly one BOS (add_bos_token=true); train
    # sequences must match. Google's HF repo ships add_bos_token=False.
    tok.add_bos_token = True
    print(f"add_bos_token: {getattr(tok, 'add_bos_token', 'n/a')}")
    collate.pad_id = tok.pad_token_id or tok.eos_token_id

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb,
        device_map={"": 0},
        attn_implementation="eager",
    )
    model.config.use_cache = False

    lora = LoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
    model.enable_input_require_grads()

    ds = PilotDataset(tok)
    print(f"Dataset: {len(ds)} examples")
    if len(ds) < 10:
        sys.exit("Dataset too small")

    args = TrainingArguments(
        output_dir="/tmp/e2b_ckpt",
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=1.5e-4,
        lr_scheduler_type="cosine",
        logging_steps=1,
        save_strategy="no",
        fp16=True,
        report_to=[],
        seed=42,
    )
    Trainer(model=model, args=args, train_dataset=ds, data_collator=collate).train()

    model.save_pretrained(ADAPTER_DIR)
    tok.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to {ADAPTER_DIR}")

    # Merge in fp16 (cannot merge into a 4-bit base): reload clean base.
    del model
    gc.collect()
    torch.cuda.empty_cache()
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map={"": 0})
    merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
    merged.config.use_cache = True

    quick_eval(merged, tok)

    merged.save_pretrained(MERGED_DIR)
    tok.save_pretrained(MERGED_DIR)
    del merged, base
    gc.collect()
    torch.cuda.empty_cache()

    r = subprocess.run(
        [sys.executable, os.path.join(LLAMA_CPP, "convert_hf_to_gguf.py"),
         MERGED_DIR, "--outfile", F16_GGUF, "--outtype", "f16"],
    )
    if r.returncode != 0:
        sys.exit(f"GGUF conversion failed (rc={r.returncode})")
    print(f"\nF16 GGUF ready: {F16_GGUF} ({os.path.getsize(F16_GGUF) / 1e9:.2f} GB)")
    print("NEXT: run the quantize cell -> /kaggle/working/gemma-4-E2B-pilot-q4_0.gguf")


if __name__ == "__main__":
    main()


In [ ]:
!python train_kaggle.py

In [ ]:
# Builds only llama-quantize (CPU, no CUDA needed). ~10-15 min.
!cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF \
  && cmake --build /tmp/llama.cpp/build --target llama-quantize -j4

In [ ]:
!/tmp/llama.cpp/build/bin/llama-quantize /tmp/e2b_pilot/pilot_f16.gguf \
  /kaggle/working/gemma-4-E2B-pilot-q4_0.gguf q4_0
!ls -la /kaggle/working